## Load Dataset (28x28 image)

In [37]:
# import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms
import random

In [38]:
transform = transforms.Compose([ #Transform image to 1 channel
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1), #channel = 1 (b&w)
    transforms.ToTensor(), #  Float [0, 1]
    transforms.Normalize((0.5,), (0.5,))  
])

In [39]:
dataset = ImageFolder(root="processed", transform=transform)

indices_empty = [i for i, (_, y) in enumerate(dataset.samples) if y == 0]
indices_occupied = [i for i, (_, y) in enumerate(dataset.samples) if y == 1]

N = 3000
random.seed(42)

selected_indices = (
    random.sample(indices_empty, N) +
    random.sample(indices_occupied, N)
)

small_dataset = Subset(dataset, selected_indices)

print("Total samples:", len(small_dataset))

Total samples: 6000


In [40]:
# split dataset
train_len = int(0.8 * len(small_dataset))
test_len = len(small_dataset) - train_len

train_dataset, test_dataset = random_split(
    small_dataset, [train_len, test_len]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle = False)

print (len(train_dataset), len(test_dataset))


4800 1200


## Bnn model

In [23]:
# 1. Hàm kích hoạt
class SignWithSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        # Biến thành +1 hoặc -1 
        return torch.where(x >= 0, torch.tensor(1.0, device=x.device), torch.tensor(-1.0, device=x.device))
        
    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[x.abs() > 1.0] = 0 
        return grad_input

In [24]:
# 2. Build Bnn core
class BinConv2d(nn.Conv2d):
    def forward(self, x):
        bin_weight = SignWithSTE.apply(self.weight)
        # Tích chập <-> Xnor + Popcount
        return F.conv2d(x, bin_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

class BinLinear(nn.Linear):
    def forward(self, x):
        bin_weight = SignWithSTE.apply(self.weight)
        return F.linear(x, bin_weight, self.bias)

In [29]:
class BNN_Verilog_Exact_V2(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 16 channel with kernel size 3x3, stride = 2 to down size
        self.conv1 = BinConv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = BinConv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = BinConv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv_out = nn.Conv2d(64, 1, kernel_size=1, stride=1, padding=0, bias=False)
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = SignWithSTE.apply(x) 

        x = self.conv2(x)
        x = self.bn2(x)
        x = SignWithSTE.apply(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = SignWithSTE.apply(x)

        x = self.conv_out(x)
        
        # Accumlulater
        x = x.mean(dim=[2, 3]) 
        
        return x
        

In [30]:
bnn_model = BNN_Verilog_Exact_V2()
criterion = nn.BCEWithLogitsLoss()
optimizer_bnn = torch.optim.Adam(bnn_model.parameters(), lr=1e-3)

print("BẮT ĐẦU TRAIN BNN...")
for epoch in range(10): # Có thể tăng lên 10-15 epoch nếu máy chạy nhanh
    bnn_model.train()
    total_loss = 0

    for x, y in train_loader:
        y = y.float().unsqueeze(1)

        optimizer_bnn.zero_grad()
        out = bnn_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer_bnn.step()

        total_loss += loss.item()

    print(f"[BNN] Epoch {epoch}: loss = {total_loss:.4f}")

BẮT ĐẦU TRAIN BNN...
[BNN] Epoch 0: loss = 155.6987
[BNN] Epoch 1: loss = 111.1842
[BNN] Epoch 2: loss = 99.0142
[BNN] Epoch 3: loss = 98.9457
[BNN] Epoch 4: loss = 102.5205
[BNN] Epoch 5: loss = 96.6077
[BNN] Epoch 6: loss = 92.0723
[BNN] Epoch 7: loss = 82.8629
[BNN] Epoch 8: loss = 79.1653
[BNN] Epoch 9: loss = 73.9129


In [32]:
# 4. Evaluate model at accuracy
def evaluate_bnn_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in dataloader:
            y = y.float().unsqueeze(1)
            out = model(x)
            
            pred = (out >= 0.0).float() 

            correct += (pred == y).sum().item()
            total += y.numel()

    acc = correct / total
    return acc

In [33]:
bnn_acc = evaluate_bnn_accuracy(bnn_model, test_loader)
print(f"BNN Accuracy: {bnn_acc*100:.2f}%")

BNN Accuracy: 93.83%


## Cnn model

In [35]:
class CNN_Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2) 
        
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2) 

        self.relu = nn.ReLU()
        self.fc = nn.Linear(16 * 7 * 7, 1) 

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [41]:
cnn = CNN_Baseline()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

for epoch in range(10):
    cnn.train()
    total_loss = 0

    for x, y in train_loader:
        y = y.float().unsqueeze(1)

        optimizer.zero_grad()
        out = cnn(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[CNN] Epoch {epoch}: loss = {total_loss:.4f}")

[CNN] Epoch 0: loss = 99.7623
[CNN] Epoch 1: loss = 62.0265
[CNN] Epoch 2: loss = 60.4778
[CNN] Epoch 3: loss = 56.4753
[CNN] Epoch 4: loss = 54.4087
[CNN] Epoch 5: loss = 52.8260
[CNN] Epoch 6: loss = 51.9958
[CNN] Epoch 7: loss = 51.4807
[CNN] Epoch 8: loss = 50.0659
[CNN] Epoch 9: loss = 48.2505


In [42]:
def evaluate_accuracy(model, dataloader):
    model.eval() 
    correct = 0
    total = 0

    with torch.no_grad(): 
        for x, y in dataloader:
            y = y.float().unsqueeze(1)
            out = model(x)
            
            pred = (out >= 0.0).float() 

            correct += (pred == y).sum().item()
            total += y.numel()

    acc = correct / total
    return acc
cnn_acc = evaluate_accuracy(cnn, test_loader)
print(f"CNN Accuracy: {cnn_acc*100:.2f}%")

CNN Accuracy: 96.25%
